In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
from PyQt6.QtGui.QRawFont import weight
from sympy.abc import lamda

from torch.utils import data
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import datasets,transforms, utils
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
USE_MPS=torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')
from pathlib import Path
from PIL import Image
import torch

class F_dataset(Dataset):
    def __init__(self,dir,transform=None):
        self.img_dir=dir
        self.img_f=list(self.img_dir.glob('*.jpg'))
        if len(self.img_f)==0:
            print("로드할 파일이 없습니다.")
        self.transform=transform
    def __len__(self):
        return len(self.img_f)
    def __getitem__(self,idx):
        img_path=self.img_f[idx]

        image=Image.open(img_path).convert('RGB')
        
        #내용 변경 파악
        label_name=img_path.stem.split('.')[0].lower() #cat.0.jpg
        if label_name=='cat':
            label=0
        elif label_name=='dog':
            label=1
        else:
            raise ValueError(f'파일 클래스 내용이 없는 파일이 로드 되었습니다{img_path.name}')

        if self.transform:
            image=self.transform(image)
            
        return image,label

In [ ]:
from torchvision import transforms,datasets#데이터 전처리 목적
from torch.utils.data import DataLoader
기준경로=base_dir
이미지크기=128
배치크기=32
transform=transforms.Compose([
    #데이터 증강
    #이미지 크기 리사이즈
    transforms.Resize((이미지크기,이미지크기)),
    #텐서화
    transforms.ToTensor(),
    #스케일정리
    #이미지net의 데이터 기본 처리 방식
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
#eval_transform
tr_ds=datasets.ImageFolder(기준경로/'train',transform=transform)
tt_ds=datasets.ImageFolder(기준경로/'test',transform=transform)
val_ds=datasets.ImageFolder(기준경로/'validation',transform=transform)

tr_ds_loader=DataLoader(tr_ds,batch_size=배치크기,shuffle=True)
tt_ds_loader=DataLoader(tt_ds,batch_size=배치크기,shuffle=True)
val_ds_loader=DataLoader(val_ds,batch_size=배치크기,shuffle=True)

In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
import json
from PIL import Image

from torch.utils import data
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import datasets,transforms, utils, models
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
USE_CUDA=torch.cuda.is_available()
DEVICE=torch.device('cuda' if USE_CUDA else 'cpu')

In [ ]:
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

In [ ]:
class BaseTransform:
    def __init__(self,resize,mean,std):
        self.base_transfrom=transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
    def __call__(self,img):
        return self.base_transfrom(img)

In [ ]:
img_data=Image.open('./data/hymenoptera_data/train/ants/0013035.jpg')
plt.imshow(img_data)
plt.show()

In [ ]:
resize=224
mean=(0.485,0.456,0.406)
std=(0.229,0.224,0.225)
transform=BaseTransform(resize,mean,std)
img_tr=transform(img_data)

In [ ]:
v_img_tr=img_tr.numpy().transpose((1,2,0))

In [ ]:
v_img_tr.shape,img_tr.shape

In [ ]:
v_img_tr=np.clip(v_img_tr,0,1)

In [ ]:
plt.imshow(v_img_tr)
plt.show()

In [ ]:
ILSVRC_class_label=json.load(open('./data/imagenet_class_index.json','r'))
ILSVRC_class_label

# 전이학습

In [ ]:
class Make_dataset_Transform_tr:
    def __init__(self,resize,mean,std):
        self.base_transform=transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ])
    def __call__(self,img):
        return self.base_transform(img)
        
class Make_dataset_Transform_val:
    def __init__(self,resize,mean,std):
        self.base_transform=transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        
    def __call__(self,img):
        return self.base_transform(img)
resize=224
mean=(0.485,0.456,0.406)
std=(0.229,0.224,0.225)
tr_base_transform=Make_dataset_Transform_tr(resize,mean,std)
val_base_transform=Make_dataset_Transform_val(resize,mean,std)
import pathlib
기준경로=pathlib.Path('./data/hymenoptera_data')
tr_ds=datasets.ImageFolder(기준경로/'train',transform=tr_base_transform)
val_ds=datasets.ImageFolder(기준경로/'val',transform=val_base_transform)

In [ ]:
class Make_dataset_Transform_tr:
    def __init__(self,resize,mean,std):
        self.base_transform=transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ])
    def __call__(self,img):
        return self.base_transform(img)
        
class Make_dataset_Transform_val:
    def __init__(self,resize,mean,std):
        self.base_transform=transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        
    def __call__(self,img):
        return self.base_transform(img)
resize=224
mean=(0.485,0.456,0.406)
std=(0.229,0.224,0.225)
tr_base_transform=Make_dataset_Transform_tr(resize,mean,std)
val_base_transform=Make_dataset_Transform_val(resize,mean,std)
import pathlib
기준경로=pathlib.Path('./data/hymenoptera_data')
tr_ds=datasets.ImageFolder(기준경로/'train',transform=tr_base_transform)
val_ds=datasets.ImageFolder(기준경로/'val',transform=val_base_transform)

In [ ]:
import pathlib
기준경로=pathlib.Path('./data/hymenoptera_data')
tr_ds=datasets.ImageFolder(기준경로/'train',transform=tr_base_transform)
val_ds=datasets.ImageFolder(기준경로/'val',transform=val_base_transform)

In [ ]:
BATCH_SIZE=32
tr_ds_loader=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True)
val_ds_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=True) 

In [ ]:
x,y=next(iter(val_ds_loader))
x.shape

In [ ]:
y

In [ ]:
class Make_dataset_Transform:
    def __init__(self,resize,mean,std):
        self.base_transform={
            'train':transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ]),
            'val':transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        }
    def __call__(self,phase='train'):
        return self.base_transform[phase]

In [ ]:
class A:
    pass
a=A()
def f(x):
    x()

f(a)

In [ ]:
class Make_dataset_Transform:
    def __init__(self,resize,mean,std):
        self.base_transform={
            'train':transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ]),
            'val':transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        }
    def __call__(self,img,phase='train'):
        return self.base_transform[phase](img)

In [ ]:
resize=224
mean=(0.485,0.456,0.406)
std=(0.229,0.224,0.225)
transform=Make_dataset_Transform(resize,mean,std)

In [ ]:
import os.path as osp
import glob
def Make_data_path_list(root_path='./data/hymenoptera_data/',phase='train'):
    target_path=osp.join(root_path+phase+'/**/*.jpg')
    out_list=[]
    for path in glob.glob(target_path):
        out_list.append(path)
    return out_list
Make_data_path_list() 

In [ ]:
class Make_dataset(Dataset):
    def __init__(self,f_list,transform,phase='train'):
        self.f_list=f_list
        self.transform=transform
        self.phase=phase
    def __len__(self):
        return len(self.f_list)
    def __getitem__(self,idex):
        img_path=self.f_list[idex]
        img=Image.open(img_path)
        img_tr=self.transform(img,self.phase)
        if  self.phase == 'train':
            label=img_path[30:34]
        elif  self.phase == 'val':
            label=img_path[28:32]
        if label=='ants':
            label=0
        elif label=='bees':
            label=1
        return img_tr,label
tr_ds=Make_dataset()        

In [ ]:
resize=224
mean=(0.485,0.456,0.406)
std=(0.229,0.224,0.225)
transform=Make_dataset_Transform(resize,mean,std)
기준경로='./data/hymenoptera_data/'
tr_list=Make_data_path_list(기준경로,'train')
val_list=Make_data_path_list(기준경로,'val')
tr_ds=Make_dataset(tr_list,transform,'train')
val_ds=Make_dataset(val_list,transform,'val')

# 데이터 로드 구조 정리

In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
import json
from PIL import Image

import pathlib

from torch.utils import data
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import datasets,transforms, utils, models
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
USE_MPS=torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

In [ ]:
#데이터 로더 구조 정의
class Make_dataset_Transform:
    def __init__(self,resize,mean,std):
        self.base_transform={
            'train':transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ]),
            'val':transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기           
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        }
    def __call__(self,phase='train'):
        return self.base_transform[phase]

In [ ]:
#상수 정의
resize=224
mean=(0.485,0.456,0.406) # 이미지넷 데이터의 표준화 값
std=(0.229,0.224,0.225) # 이미지넷 데이터의 표준화 값
"""
mean, std
전체 데이터 관점에서 평균을 도출하거나,
데이터 전처리 방식에 따라 결정
ex) [-1,1] mean, std = (0.5, 0.5, 0.5),(0.5, 0.5, 0.5)
전이 학습을 활용 한다면 가급적 사용 모델의 표준화를 사용할 것
*이미지는 반드시 체널 단위 연산*
"""
BATCH_SIZE=32

#데이터 로더 정의
transform=Make_dataset_Transform(resize,mean,std)
기준경로=pathlib.Path('../../data/hymenoptera_data')
tr_ds=datasets.ImageFolder(기준경로/'train',transform=transform('train'))
val_ds=datasets.ImageFolder(기준경로/'val',transform=transform('val'))
tr_ds_loader=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True)
val_ds_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
tr_x, tr_y = next(iter(tr_ds_loader))
val_x, val_y = next(iter(val_ds_loader))

In [ ]:
from torchvision.models import vgg16, VGG16_Weights
vgg16_m = vgg16(weights = VGG16_Weights.IMAGENET1K_V1)
vgg16_m

In [ ]:
for p in vgg16_m.features.parameters():
    p.requires_grad=False

In [ ]:
class_n = 2
classifier_in_n = vgg16_m.classifier[-1].in_features
vgg16_m.classifier[-1] = nn.Linear(classifier_in_n, class_n)

In [ ]:
vgg16_m

In [ ]:
vgg16_m = vgg16_m.to(DEVICE)
criterion = nn.CrossEntropyLoss()
opt = optim.Adam(filter(lambda p: p.requires_grad, vgg16_m.parameters()), lr = 0.0001)

In [ ]:
def run_epoch(m, loder, train=True):
    m.train(train)
    total_loss, total_correct, total = 0.0, 0, 0
    for x,y in loder:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train:
            opt.zero_grad()
        with torch.set_grad_enabled(train):
            output = m(x)
            loss = criterion(output, y)
            if train:
                loss.backward()
                opt.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (output.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total

tr_loss, tr_acc = run_epoch(vgg16_m, tr_ds_loader, True)
val_loss, val_acc = run_epoch(vgg16_m, val_ds_loader, False)

print(f'tr_loss: {tr_loss:.4f}, tr_acc: {tr_acc:.4f} | val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}')

# stanford dog 종 분류기를 전이 학습을 이용하여 완성코드를 완성하시오, 단 학습 횟수는 5회로 제한한다.

In [11]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
import json
from PIL import Image
import os
import shutil
import glob
from sklearn.model_selection import train_test_split
from pathlib import Path

from torch.utils import data
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import datasets,transforms, utils, models
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
USE_MPS=torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

In [12]:
# --- 1. 경로 설정 ---
# 원본 Stanford Dogs Dataset 경로 (Images 폴더가 있는 경로)
original_dataset_dir = '../../data/stanford_dogs/images/Images'

# 데이터를 새로 저장할 기본 경로
base_dir = Path('../../data/stanford_dogs_split')
if not os.path.exists(base_dir):
    os.mkdir(base_dir)

# 훈련, 검증 디렉토리 경로 설정
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

if not os.path.exists(train_dir):
    os.mkdir(train_dir)
if not os.path.exists(validation_dir):
    os.mkdir(validation_dir)


# --- 2. 품종 목록 가져오기 ---
# 원본 데이터셋의 품종 폴더 목록을 가져옵니다.
breeds = os.listdir(original_dataset_dir)


# --- 3. 훈련/검증 데이터 분할 및 복사 ---
print("데이터 분할 및 복사를 시작합니다...")

for breed_folder in breeds:
    # 각 품종별로 훈련 및 검증 폴더 생성
    train_breed_dir = os.path.join(train_dir, breed_folder)
    validation_breed_dir = os.path.join(validation_dir, breed_folder)
    if not os.path.exists(train_breed_dir):
        os.mkdir(train_breed_dir)
    if not os.path.exists(validation_breed_dir):
        os.mkdir(validation_breed_dir)

    # 원본 품종 폴더의 모든 이미지 파일 경로를 가져오기
    src_dir = os.path.join(original_dataset_dir, breed_folder)
    image_files = glob.glob(os.path.join(src_dir, '*.jpg'))

    # train_test_split을 사용하여 8:2 비율로 파일 목록 분할
    train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

    # 훈련 데이터 복사
    for f in train_files:
        shutil.copy(f, os.path.join(train_breed_dir, os.path.basename(f)))

    # 검증 데이터 복사
    for f in val_files:
        shutil.copy(f, os.path.join(validation_breed_dir, os.path.basename(f)))

print(f"데이터 분할 완료! '{base_dir}' 폴더를 확인해주세요.")

데이터 분할 및 복사를 시작합니다...
데이터 분할 완료! '../../data/stanford_dogs_split' 폴더를 확인해주세요.


In [13]:
#데이터 로더 구조 정의
class Make_dataset_Transform:
    def __init__(self,resize,mean,std):
        self.base_transform={
            'train':transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize,scale=(0.5,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ]),
            'val':transforms.Compose([
            transforms.Resize(resize),#짧은변 기준 리사이즈
            transforms.CenterCrop(resize),#이미지 중앙을 resize 로 자르기
            transforms.ToTensor(),
            transforms.Normalize(mean,std)
        ])
        }
    def __call__(self,phase='train'):
        return self.base_transform[phase]

In [14]:
#상수 정의
resize=224
mean=(0.485,0.456,0.406) # 이미지넷 데이터의 표준화 값
std=(0.229,0.224,0.225) # 이미지넷 데이터의 표준화 값
"""
mean, std
전체 데이터 관점에서 평균을 도출하거나,
데이터 전처리 방식에 따라 결정
ex) [-1,1] mean, std = (0.5, 0.5, 0.5),(0.5, 0.5, 0.5)
전이 학습을 활용 한다면 가급적 사용 모델의 표준화를 사용할 것
*이미지는 반드시 체널 단위 연산*
"""
BATCH_SIZE=128

#데이터 로더 정의
transform=Make_dataset_Transform(resize,mean,std)
기준경로=Path('../../data/stanford_dogs_split')
tr_ds=datasets.ImageFolder(기준경로/'train',transform=transform('train'))
val_ds=datasets.ImageFolder(기준경로/'validation',transform=transform('val'))
tr_ds_loader=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True)
val_ds_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)

In [15]:
tr_x, tr_y = next(iter(tr_ds_loader))
val_x, val_y = next(iter(val_ds_loader))

In [16]:
from torchvision.models import vgg16, VGG16_Weights
vgg16_m = vgg16(weights = VGG16_Weights.IMAGENET1K_V1)

In [17]:
for p in vgg16_m.features.parameters():
    p.requires_grad=False

In [18]:
class_n = len(tr_ds.classes)
classifier_in_n = vgg16_m.classifier[-1].in_features
vgg16_m.classifier[-1] = nn.Linear(classifier_in_n, class_n)

In [19]:
vgg16_m = vgg16_m.to(DEVICE)
criterion = nn.CrossEntropyLoss()
opt = optim.Adam(filter(lambda p: p.requires_grad, vgg16_m.parameters()), lr = 0.0001)

In [20]:
def run_epoch(m, loder, train=True):
    m.train(train)
    total_loss, total_correct, total = 0.0, 0, 0
    for x,y in loder:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train:
            opt.zero_grad()
        with torch.set_grad_enabled(train):
            output = m(x)
            loss = criterion(output, y)
            if train:
                loss.backward()
                opt.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (output.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total

num_epochs = 10
for epoch in range(num_epochs):
    tr_loss, tr_acc = run_epoch(vgg16_m, tr_ds_loader, True)
    val_loss, val_acc = run_epoch(vgg16_m, val_ds_loader, False)

    print(f'Epoch {epoch+1}/{num_epochs} | tr_loss: {tr_loss:.4f}, tr_acc: {tr_acc:.4f} | val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}')

Epoch 1/10 | tr_loss: 1.7191, tr_acc: 0.5573 | val_loss: 0.6564, val_acc: 0.7982
Epoch 2/10 | tr_loss: 0.7934, tr_acc: 0.7563 | val_loss: 0.5656, val_acc: 0.8229
Epoch 3/10 | tr_loss: 0.6649, tr_acc: 0.7911 | val_loss: 0.5607, val_acc: 0.8205
Epoch 4/10 | tr_loss: 0.5686, tr_acc: 0.8203 | val_loss: 0.5659, val_acc: 0.8174
Epoch 5/10 | tr_loss: 0.5040, tr_acc: 0.8365 | val_loss: 0.5576, val_acc: 0.8220
Epoch 6/10 | tr_loss: 0.4427, tr_acc: 0.8588 | val_loss: 0.5534, val_acc: 0.8256
Epoch 7/10 | tr_loss: 0.3963, tr_acc: 0.8689 | val_loss: 0.5461, val_acc: 0.8321
Epoch 8/10 | tr_loss: 0.3678, tr_acc: 0.8806 | val_loss: 0.5492, val_acc: 0.8289
Epoch 9/10 | tr_loss: 0.3308, tr_acc: 0.8914 | val_loss: 0.5508, val_acc: 0.8265
Epoch 10/10 | tr_loss: 0.3105, tr_acc: 0.8980 | val_loss: 0.5712, val_acc: 0.8212
